In [6]:
import yfinance as yf

sp = yf.download("^GSPC", start="1989-01-01", end="2026-01-01", auto_adjust=False)
sp.columns = sp.columns.get_level_values(0)  # deja una sola fila de encabezados
sp.to_csv("gspc_diario.csv")

print(sp.shape)
print(sp.head(3))
print(sp.tail(3))

[*********************100%***********************]  1 of 1 completed

(9319, 6)
Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
1989-01-03  275.309998  275.309998  277.720001  273.809998  277.720001   
1989-01-04  279.429993  279.429993  279.750000  275.309998  275.309998   
1989-01-05  280.010010  280.010010  281.510010  279.429993  279.429993   

Price          Volume  
Date                   
1989-01-03  128500000  
1989-01-04  149700000  
1989-01-05  174040000  
Price         Adj Close        Close         High          Low         Open  \
Date                                                                          
2025-12-29  6905.740234  6905.740234  6920.209961  6888.759766  6903.600098   
2025-12-30  6896.240234  6896.240234  6913.250000  6893.470215  6900.439941   
2025-12-31  6845.500000  6845.500000  6901.419922  6844.549805  6898.819824   

Price           Volume  
Date                    
2025-12-29  3541750000  
2025-12-30  3309930000  
202

In [ ]:
"""
Tarea 2 AFML - Regresion logistica de referencia
Objetivo: predecir caida10 (caida >=10% del S&P 500 en los proximos 3 meses)
"""
import time
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
import statsmodels.api as sm

# ---------------------------------------------------------------
# 1. Datos
# ---------------------------------------------------------------
df = pd.read_csv("base_tarea2_caidas_sp500.csv", index_col="fecha")
df = df.dropna(subset=["caida10"])  # quita los ultimos 3 meses (objetivo desconocido)

PREDICTORES = ["cape_lag3", "mom_12m", "vix_fin", "baa10y_fin", "t10y3m_fin"]
X = df[PREDICTORES]
y = df["caida10"].astype(int)

# ---------------------------------------------------------------
# 2. Separacion entrenamiento / prueba 
# ---------------------------------------------------------------
INICIO_PRUEBA = "2015-01"
HORIZONTE = 3

es_train = X.index < INICIO_PRUEBA
idx_train = X.index[es_train][:-HORIZONTE]
idx_test = X.index[~es_train]

X_train, y_train = X.loc[idx_train], y.loc[idx_train]
X_test, y_test = X.loc[idx_test], y.loc[idx_test]

print(f"Entrenamiento: {idx_train[0]} a {idx_train[-1]} ({len(idx_train)} meses, {y_train.sum()} caidas)")
print(f"Prueba:        {idx_test[0]} a {idx_test[-1]} ({len(idx_test)} meses, {y_test.sum()} caidas)\n")

# ---------------------------------------------------------------
# 3. Modelo: estandarizar + logistica
# ---------------------------------------------------------------
pipe = Pipeline([
    ("escalar", StandardScaler()),
    ("logit", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

# ---------------------------------------------------------------
# 4. Validacion (solo dentro del entrenamiento) para elegir C
# ---------------------------------------------------------------
cv = TimeSeriesSplit(n_splits=5, gap=HORIZONTE)
grilla = {"logit__C": [0.001, 0.01, 0.1, 1, 10, 100]}

busqueda = GridSearchCV(pipe, grilla, cv=cv, scoring="roc_auc")
t0 = time.time()
busqueda.fit(X_train, y_train)
tiempo = time.time() - t0

print("AUC de validacion por valor de C (sensibilidad a hiperparametros):")
res = pd.DataFrame(busqueda.cv_results_)[["param_logit__C", "mean_test_score", "std_test_score"]]
print(res.round(3).to_string(index=False))
print(f"\nMejor C: {busqueda.best_params_['logit__C']}  |  tiempo de entrenamiento: {tiempo:.2f} s\n")

modelo = busqueda.best_estimator_

# ---------------------------------------------------------------
# 5. Metricas en entrenamiento y prueba
# ---------------------------------------------------------------
UMBRAL = 0.5  # probabilidad sobre la cual se predice "caida"

def metricas(modelo, X, y, umbral=UMBRAL):
    prob = modelo.predict_proba(X)[:, 1]
    pred = (prob >= umbral).astype(int)
    return {
        "AUC": roc_auc_score(y, prob),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
    }

m_train = metricas(modelo, X_train, y_train)
m_test = metricas(modelo, X_test, y_test)
tabla = pd.DataFrame({"Entrenamiento": m_train, "Prueba": m_test})
tabla["Brecha (train - test)"] = tabla["Entrenamiento"] - tabla["Prueba"]
print("Desempeno de la logistica:")
print(tabla.round(3).to_string())

print("\nMatriz de confusion en prueba (filas = real, columnas = predicho):")
pred_test = (modelo.predict_proba(X_test)[:, 1] >= UMBRAL).astype(int)
print(pd.DataFrame(confusion_matrix(y_test, pred_test),
                   index=["Real 0", "Real 1"], columns=["Pred 0", "Pred 1"]))

# ---------------------------------------------------------------
# 6. Interpretacion: coeficientes estandarizados
#    Signo positivo = esa variable aumenta la probabilidad de caida.
#    Magnitud = efecto de subir 1 desviacion estandar la variable.
# ---------------------------------------------------------------
coef = pd.Series(modelo.named_steps["logit"].coef_[0], index=PREDICTORES)
print("\nCoeficientes (variables estandarizadas):")
print(coef.round(3).to_string())

# Version statsmodels (sin regularizacion ni pesos) para ver significancia,
# igual que la salida de regresion vista en clases
Xs = sm.add_constant(StandardScaler().fit(X_train).transform(X_train))
logit_sm = sm.Logit(y_train.values, pd.DataFrame(Xs, columns=["const"] + PREDICTORES)).fit(disp=0)
print("\nLogit con statsmodels (entrenamiento, variables estandarizadas):")
print(logit_sm.summary2().tables[1][["Coef.", "Std.Err.", "P>|z|"]].round(3))

# Guardar resultados para la tabla comparativa final
tabla.to_csv("resultados_logistica.csv")

Entrenamiento: 1990-01 a 2014-09 (297 meses, 41 caidas)
Prueba:        2015-01 a 2025-09 (129 meses, 21 caidas)

AUC de validacion por valor de C (sensibilidad a hiperparametros):
param_logit__C  mean_test_score  std_test_score
         0.001            0.533           0.319
          0.01            0.528           0.323
           0.1            0.471           0.267
             1            0.454           0.249
            10            0.480           0.261
           100            0.469           0.266

Mejor C: 0.001  |  tiempo de entrenamiento: 0.16 s

Desempeno de la logistica:
           Entrenamiento  Prueba  Brecha (train - test)
AUC                0.773   0.631                  0.142
Precision          0.318   0.258                  0.060
Recall             0.683   0.381                  0.302
F1                 0.434   0.308                  0.126

Matriz de confusion en prueba (filas = real, columnas = predicho):
        Pred 0  Pred 1
Real 0      85      23
Real 1    

In [ ]:
"""
Tarea 2 AFML - KNN (K vecinos mas cercanos)
Objetivo: predecir caida10 (caida >=10% del S&P 500 en los proximos 3 meses)
"""
import time
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix

# ---------------------------------------------------------------
# 1. Datos (igual que la logistica)
# ---------------------------------------------------------------
df = pd.read_csv("base_tarea2_caidas_sp500.csv", index_col="fecha")
df = df.dropna(subset=["caida10"])

PREDICTORES = ["cape_lag3", "mom_12m", "vix_fin", "baa10y_fin", "t10y3m_fin"]
X = df[PREDICTORES]
y = df["caida10"].astype(int)

# ---------------------------------------------------------------
# 2. Particion entrenamiento / prueba (igual que la logistica)
# ---------------------------------------------------------------
INICIO_PRUEBA = "2015-01"
HORIZONTE = 3

es_train = X.index < INICIO_PRUEBA
idx_train = X.index[es_train][:-HORIZONTE]   
idx_test = X.index[~es_train]

X_train, y_train = X.loc[idx_train], y.loc[idx_train]
X_test, y_test = X.loc[idx_test], y.loc[idx_test]

print(f"Entrenamiento: {idx_train[0]} a {idx_train[-1]} ({len(idx_train)} meses, {y_train.sum()} caidas)")
print(f"Prueba:        {idx_test[0]} a {idx_test[-1]} ({len(idx_test)} meses, {y_test.sum()} caidas)\n")

# ---------------------------------------------------------------
# 3. Modelo: estandarizar + KNN
# ---------------------------------------------------------------
pipe = Pipeline([
    ("escalar", StandardScaler()),
    ("knn", KNeighborsClassifier(metric="euclidean")),
])

# ---------------------------------------------------------------
# 4. Validacion temporal para elegir K (numero de vecinos)
#    K chico = modelo flexible (riesgo de sobreajuste)
#    K grande = modelo suave (riesgo de no captar el patron)
# ---------------------------------------------------------------
cv = TimeSeriesSplit(n_splits=5, gap=HORIZONTE)
grilla = {"knn__n_neighbors": [1, 3, 5, 7, 11, 15, 21, 31, 41]}

busqueda = GridSearchCV(pipe, grilla, cv=cv, scoring="roc_auc", return_train_score=True)
t0 = time.time()
busqueda.fit(X_train, y_train)
tiempo = time.time() - t0

res = pd.DataFrame(busqueda.cv_results_)
res = res[["param_knn__n_neighbors", "mean_train_score", "mean_test_score", "std_test_score"]]
res.columns = ["K", "AUC entrenamiento", "AUC validacion", "Desv. validacion"]
print("Sensibilidad a K (AUC promedio en los 5 bloques de validacion):")
print(res.round(3).to_string(index=False))
K = busqueda.best_params_["knn__n_neighbors"]
print(f"\nMejor K: {K}  |  tiempo de busqueda y entrenamiento: {tiempo:.2f} s\n")

modelo = busqueda.best_estimator_

# ---------------------------------------------------------------
# 5. Metricas en entrenamiento y prueba
# ---------------------------------------------------------------
UMBRAL = y_train.mean()
print(f"Umbral = frecuencia de caidas en entrenamiento = {UMBRAL:.3f}\n")

def metricas(modelo, X, y, umbral):
    prob = modelo.predict_proba(X)[:, 1]
    pred = (prob >= umbral).astype(int)
    return {
        "AUC": roc_auc_score(y, prob),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
    }

m_train = metricas(modelo, X_train, y_train, UMBRAL)
m_test = metricas(modelo, X_test, y_test, UMBRAL)
tabla = pd.DataFrame({"Entrenamiento": m_train, "Prueba": m_test})
tabla["Brecha (train - test)"] = tabla["Entrenamiento"] - tabla["Prueba"]
print(f"Desempeno de KNN (K = {K}):")
print(tabla.round(3).to_string())

print("\nMatriz de confusion en prueba (filas = real, columnas = predicho):")
pred_test = (modelo.predict_proba(X_test)[:, 1] >= UMBRAL).astype(int)
print(pd.DataFrame(confusion_matrix(y_test, pred_test),
                   index=["Real 0", "Real 1"], columns=["Pred 0", "Pred 1"]))

# ---------------------------------------------------------------
# 6. Ilustracion del sobreajuste: K = 1
# ---------------------------------------------------------------
knn1 = Pipeline([("escalar", StandardScaler()),
                 ("knn", KNeighborsClassifier(n_neighbors=1))]).fit(X_train, y_train)
print("\nComparacion con K = 1 (memoriza la muestra):")
print(f"  AUC entrenamiento: {roc_auc_score(y_train, knn1.predict_proba(X_train)[:, 1]):.3f}")
print(f"  AUC prueba:        {roc_auc_score(y_test, knn1.predict_proba(X_test)[:, 1]):.3f}")

# Guardar resultados para la tabla comparativa final
tabla.to_csv("resultados_knn.csv")
res.to_csv("sensibilidad_knn.csv", index=False)

Entrenamiento: 1990-01 a 2014-09 (297 meses, 41 caidas)
Prueba:        2015-01 a 2025-09 (129 meses, 21 caidas)

Sensibilidad a K (AUC promedio en los 5 bloques de validacion):
 K  AUC entrenamiento  AUC validacion  Desv. validacion
 1              1.000           0.480             0.024
 3              0.979           0.459             0.062
 5              0.967           0.511             0.116
 7              0.947           0.538             0.154
11              0.941           0.586             0.191
15              0.927           0.612             0.199
21              0.899           0.594             0.212
31              0.868           0.577             0.211
41              0.802           0.646             0.156

Mejor K: 41  |  tiempo de busqueda y entrenamiento: 0.27 s

Umbral = frecuencia de caidas en entrenamiento = 0.138

Desempeno de KNN (K = 41):
           Entrenamiento  Prueba  Brecha (train - test)
AUC                0.839   0.619                  0.220
Precisi

In [1]:
"""
Tarea 2 AFML - SVM con kernel RBF
Objetivo: predecir caida10 (caida >=10% del S&P 500 en los proximos 3 meses)

IMPORTANTE: usa la misma base, los mismos predictores y la misma particion
que la regresion logistica. Las secciones 1 y 2 son copia exacta de ese codigo;
no modificar sin avisar al grupo.
"""
import time
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix

# ---------------------------------------------------------------
# 1. Datos (igual que la logistica)
# ---------------------------------------------------------------
df = pd.read_csv("base_tarea2_caidas_sp500.csv", index_col="fecha")
df = df.dropna(subset=["caida10"])  # quita los ultimos 3 meses (objetivo desconocido)

PREDICTORES = ["cape_lag3", "mom_12m", "vix_fin", "baa10y_fin", "t10y3m_fin"]
X = df[PREDICTORES]
y = df["caida10"].astype(int)

# ---------------------------------------------------------------
# 2. Separacion entrenamiento / prueba (igual que la logistica)
# ---------------------------------------------------------------
INICIO_PRUEBA = "2015-01"
HORIZONTE = 3

es_train = X.index < INICIO_PRUEBA
idx_train = X.index[es_train][:-HORIZONTE]
idx_test = X.index[~es_train]

X_train, y_train = X.loc[idx_train], y.loc[idx_train]
X_test, y_test = X.loc[idx_test], y.loc[idx_test]

print(f"Entrenamiento: {idx_train[0]} a {idx_train[-1]} ({len(idx_train)} meses, {y_train.sum()} caidas)")
print(f"Prueba:        {idx_test[0]} a {idx_test[-1]} ({len(idx_test)} meses, {y_test.sum()} caidas)\n")

# ---------------------------------------------------------------
# 3. Modelo: estandarizar + SVM
#    El escalador va DENTRO del pipeline: en cada vuelta de validacion se
#    ajusta solo con el pasado, sin filtrar informacion del futuro.
# ---------------------------------------------------------------
pipe = Pipeline([
    ("escalar", StandardScaler()),
    ("svm", SVC(kernel="rbf", class_weight="balanced")),
])

# ---------------------------------------------------------------
# 4. Validacion (solo dentro del entrenamiento) para elegir C y gamma
#    C: cuanto castiga equivocarse (alto = frontera ajustada, riesgo de sobreajuste)
#    gamma: que tan local es cada punto (alto = frontera muy curva, riesgo de sobreajuste)
# ---------------------------------------------------------------
cv = TimeSeriesSplit(n_splits=5, gap=HORIZONTE)
grilla = {
    "svm__C": [0.01, 0.1, 1, 10, 100],
    "svm__gamma": [0.001, 0.01, 0.1, 1],
}

busqueda = GridSearchCV(pipe, grilla, cv=cv, scoring="roc_auc")
t0 = time.time()
busqueda.fit(X_train, y_train)
tiempo = time.time() - t0  # misma definicion que en la logistica: busqueda completa + reajuste final

res = pd.DataFrame(busqueda.cv_results_)
res["C"] = res["param_svm__C"].astype(float)
res["gamma"] = res["param_svm__gamma"].astype(float)

print("AUC de validacion promedio por (C, gamma)  [filas = C, columnas = gamma]:")
print(res.pivot(index="C", columns="gamma", values="mean_test_score").round(3).to_string())
print("\nDesviacion estandar del AUC entre vueltas de validacion:")
print(res.pivot(index="C", columns="gamma", values="std_test_score").round(3).to_string())

mejor = busqueda.best_params_
print(f"\nMejor combinacion: C = {mejor['svm__C']}, gamma = {mejor['svm__gamma']}"
      f"  |  AUC de validacion = {busqueda.best_score_:.3f}"
      f"  |  tiempo de entrenamiento: {tiempo:.2f} s\n")

modelo = busqueda.best_estimator_

# ---------------------------------------------------------------
# 5. Metricas en entrenamiento y prueba
#    - AUC se calcula con decision_function (puntaje de distancia a la frontera).
#      No se usa probability=True: su calibracion interna (Platt) usa una
#      validacion cruzada que ignora el orden del tiempo.
#    - Precision, recall y F1 usan predict(), que corresponde al umbral 0 del puntaje.
# ---------------------------------------------------------------
def metricas(modelo, X, y):
    puntaje = modelo.decision_function(X)
    pred = modelo.predict(X)
    return {
        "AUC": roc_auc_score(y, puntaje),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
    }

m_train = metricas(modelo, X_train, y_train)
m_test = metricas(modelo, X_test, y_test)
tabla = pd.DataFrame({"Entrenamiento": m_train, "Prueba": m_test})
tabla["Brecha (train - test)"] = tabla["Entrenamiento"] - tabla["Prueba"]
print("Desempeno de la SVM:")
print(tabla.round(3).to_string())

print("\nMatriz de confusion en prueba (filas = real, columnas = predicho):")
print(pd.DataFrame(confusion_matrix(y_test, modelo.predict(X_test)),
                   index=["Real 0", "Real 1"], columns=["Pred 0", "Pred 1"]))

# ---------------------------------------------------------------
# 6. Diagnostico de complejidad e interpretacion
# ---------------------------------------------------------------
# Vectores de soporte: los puntos que definen la frontera. Si casi todas las
# observaciones lo son, el modelo esta "memorizando" en vez de encontrar una regla.
n_sv = modelo.named_steps["svm"].n_support_.sum()
print(f"\nVectores de soporte: {n_sv} de {len(X_train)} observaciones de entrenamiento "
      f"({n_sv / len(X_train):.0%})")

# La SVM con kernel RBF no tiene coeficientes. Se usa importancia por permutacion:
# cuanto cae el AUC al desordenar una variable. Se calcula con el modelo ya terminado.
imp = permutation_importance(modelo, X_test, y_test, scoring="roc_auc",
                             n_repeats=30, random_state=0)
imp_df = pd.DataFrame({"Caida del AUC": imp.importances_mean,
                       "Desv. est.": imp.importances_std}, index=PREDICTORES)
print("\nImportancia por permutacion (muestra de prueba):")
print(imp_df.sort_values("Caida del AUC", ascending=False).round(3).to_string())

# ---------------------------------------------------------------
# 7. Guardar resultados para la tabla comparativa final
# ---------------------------------------------------------------
tabla.to_csv("resultados_svm.csv")
pd.DataFrame({"C": [mejor["svm__C"]], "gamma": [mejor["svm__gamma"]],
              "AUC_validacion": [busqueda.best_score_], "tiempo_s": [tiempo],
              "vectores_soporte": [n_sv]}).to_csv("resultados_svm_extra.csv", index=False)

Entrenamiento: 1990-01 a 2014-09 (297 meses, 41 caidas)
Prueba:        2015-01 a 2025-09 (129 meses, 21 caidas)

AUC de validacion promedio por (C, gamma)  [filas = C, columnas = gamma]:
gamma   0.001  0.010  0.100  1.000
C                                 
0.01    0.532  0.535  0.694  0.656
0.10    0.532  0.535  0.700  0.655
1.00    0.529  0.527  0.535  0.578
10.00   0.522  0.432  0.500  0.569
100.00  0.452  0.441  0.485  0.573

Desviacion estandar del AUC entre vueltas de validacion:
gamma   0.001  0.010  0.100  1.000
C                                 
0.01    0.318  0.311  0.191  0.141
0.10    0.318  0.311  0.194  0.146
1.00    0.320  0.305  0.137  0.209
10.00   0.327  0.194  0.226  0.190
100.00  0.206  0.204  0.231  0.178

Mejor combinacion: C = 0.1, gamma = 0.1  |  AUC de validacion = 0.700  |  tiempo de entrenamiento: 0.27 s

Desempeno de la SVM:
           Entrenamiento  Prueba  Brecha (train - test)
AUC                0.824   0.625                  0.199
Precision          0.322